# Muraqam (مُرقّم) — diacritization + weighted 3-model ensemble

Your best stack (**MSA + AraBERT + arabpunc, arabpunc weighted 2×**, AMP + early stopping) with an **optional tashkeel-restoration preprocessing** step.

**Why:** the KSAA paper found diacritics *help* punctuation prediction, but our text is bare. We restore tashkeel with a pretrained diacritizer, then run the same pipeline on the diacritized input. **Alignment guard:** we only ADD tashkeel — never change letters or word boundaries — and fall back to the bare word on any drift, so the metric's word-gap alignment is always preserved.

Toggle `USE_DIACRITIZATION` to A/B diacritized vs bare input against the host metric.

> Internet ON + GPU.

In [ ]:
# =========================================================================
#  Muraqam (مُرقّم) — Arabic Punctuation Restoration :: SOTA pipeline
#  Hybrid: deterministic honorific rules + multi-LABEL Arabic-encoder ensemble
#  Metric: macro-F1 over 7 marks ( . ، ؟ ! : ؛ - ), multi-label per gap.
# =========================================================================
!pip install -q transformers torch

import os, re, random, math, json
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer, AutoModel

SEED = 2026
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
device = "cuda" if torch.cuda.is_available() else "cpu"

# --- data paths (Kaggle) ---
TRAIN_CSV = "/kaggle/input/competitions/muraqqamchallenge/train.csv"   # adjust to your dataset slug
TEST_CSV  = "/kaggle/input/competitions/muraqqamchallenge/test.csv"
if not os.path.exists(TRAIN_CSV):                         # fallback for local
    TRAIN_CSV = "train.csv"; TEST_CSV = "test.csv"

# --- the 7 scored marks, fixed canonical order ---
MARKS = ['.', '،', '؟', '!', ':', '؛', '-']
M2I = {m: i for i, m in enumerate(MARKS)}
NUM_MARKS = len(MARKS)

# --- ensemble of MSA-native encoders (drop mBERT; it is the weakest for Arabic) ---
# 3-model ensemble: CAMeLBERT-MSA + AraBERT + arabpunc (warm-start).
# arabpunc is the strongest solo member, so it gets a heavier weight in the
# averaged probabilities (MODEL_WEIGHTS, aligned by index with MODELS).
MODELS = [
    "CAMeL-Lab/bert-base-arabic-camelbert-msa",
    "aubmindlab/bert-base-arabertv2",
    "makdadTaleb/arabic-punctuation-arabert",     # strongest -> more weight
]
MODEL_WEIGHTS = [1.0, 1.0, 2.0]                    # arabpunc weighted 2x

# --- training config ---
CHUNK_WORDS = 180     # words per window (docs are long -> sliding window)
STRIDE      = 140     # overlap = CHUNK_WORDS - STRIDE = 40 words
MAX_LEN     = 320     # subword cap per window
BATCH       = 8
EPOCHS      = 10
LR          = 2e-5
FOCAL_GAMMA = 2.0     # focal loss: focus on hard/rare marks (macro-F1 needs rares)
VAL_FRAC    = 0.10
USE_AMP           = True
EARLY_STOP_PATIENCE = 5
MAX_EPOCHS_CAP    = 20

# --- honorifics that get wrapped as -X- (data: ﷺ is 90%+ wrapped) ---
HONORIFICS = {"ﷺ"}
FORCE_HONORIFIC_RULE = True
print("device:", device, "| models:", MODELS)

## 1. Host metric

In [ ]:
# Host metric (verbatim) — for trustworthy LOCAL validation before you submit.
class ParticipantVisibleError(Exception):
    pass

"""
Kaggle metric for Arabic Punctuation Restoration.

Conventions:
    - `solution` is the full test CSV, including the hidden gold column.
    - `submission` is the competitor's CSV.
    - Both have a row_id column (already aligned & sorted by Kaggle).
    - `solution` has a column `raw` with the unpunctuated input and a column
      `gold` with the reference punctuated string.
    - `submission` has a column `prediction` with the competitor's
      punctuated string.

Scoring:
    Macro-F1 over the 7 Arabic sentence-punctuation classes
    ( . ، ؟ ! : ؛ - ), EXCLUDING the "no punctuation" class.

"""

import re
from typing import Optional

import pandas as pd
from sklearn.metrics import f1_score
from sklearn.preprocessing import MultiLabelBinarizer

# ---------------------------------------------------------------------------
# Classical Arabic punctuation-restoration set. Anything outside this is
# treated as word content (e.g. parentheses, quotes, numbers) and is NOT
# scored. Competitors must preserve those characters in their predictions.
# ---------------------------------------------------------------------------
VALID_SYMBOLS = set('.،؟!:؛-')


def _tokenize_gold(text: str):
    """
    Split a (gold or prediction) string into (leading_gap, [(word, trailing_gap), ...]).
    A "word" is a maximal run of non-whitespace, non-whitelist chars.
    A "gap" is any run of whitelist chars between words.
    """
    leading_gap_chars = []
    pairs = []
    current_word_chars = []
    in_word = False

    for ch in text:
        if ch.isspace():
            if in_word:
                pairs.append([''.join(current_word_chars), []])
                current_word_chars = []
                in_word = False
            continue

        if ch in VALID_SYMBOLS:
            if in_word:
                pairs.append([''.join(current_word_chars), [ch]])
                current_word_chars = []
                in_word = False
            else:
                if pairs:
                    pairs[-1][1].append(ch)
                else:
                    leading_gap_chars.append(ch)
            continue

        if not in_word:
            in_word = True
            current_word_chars = [ch]
        else:
            current_word_chars.append(ch)

    if in_word:
        pairs.append([''.join(current_word_chars), []])

    return ''.join(leading_gap_chars), [(w, ''.join(g)) for (w, g) in pairs]


def _extract_labels(raw_text: str, generated_text: str, role: str):
    """
    Align `generated_text` to `raw_text` word-by-word and return one list of
    symbols per word, representing the punctuation that appears in the gap
    after each word.

    Raises ValueError on any structural mismatch.
    """
    if raw_text is None or generated_text is None:
        raise ValueError(f"[{role}] text is empty or null")

    raw_words = str(raw_text).strip().split()
    if not raw_words:
        raise ValueError(f"[{role}] raw text contains no words")

    _, pairs = _tokenize_gold(str(generated_text))
    gen_words = [w for (w, _) in pairs]

    if len(gen_words) != len(raw_words):
        raise ValueError(
            f"[{role}] word-count mismatch: raw has {len(raw_words)} words, "
            f"{role} has {len(gen_words)}"
        )

    for i, (rw, gw) in enumerate(zip(raw_words, gen_words)):
        if rw != gw:
            raise ValueError(
                f"[{role}] word mismatch at position {i}: "
                f"raw='{rw}' vs {role}='{gw}'"
            )

    labels = []
    for _, gap in pairs:
        syms = [c for c in gap if c in VALID_SYMBOLS]
        labels.append(syms if syms else ['0'])

    return labels


def score(
    solution: pd.DataFrame,
    submission: pd.DataFrame,
    row_id_column_name: str,
    raw_column_name: str = 'text',
    gold_column_name: str = 'final_text',
    prediction_column_name: str = 'final_text',
) -> float:
    """
    Returns macro-F1 over the 7 Arabic punctuation
    classes, excluding the "no punctuation" class.
    """
    # --- 0. Drop row_id; Kaggle has already aligned the two frames -----------
    del solution[row_id_column_name]
    del submission[row_id_column_name]

    # --- 1. Column presence -------------------------------------------------
    for col in (raw_column_name, gold_column_name):
        if col not in solution.columns:
            # Organizer-side problem — hidden from competitor.
            raise RuntimeError(f"Solution is missing column '{col}'")
    if prediction_column_name not in submission.columns:
        raise ParticipantVisibleError(
            f"Submission is missing column '{prediction_column_name}'"
        )

    # --- 2. Length match ----------------------------------------------------
    if len(solution) != len(submission):
        raise ParticipantVisibleError(
            f"Submission has {len(submission)} rows, expected {len(solution)}"
        )

    # --- 3. Extract labels row-by-row --------------------------------------
    true_labels = []
    pred_labels = []

    raws = solution[raw_column_name].tolist()
    golds = solution[gold_column_name].tolist()
    preds = submission[prediction_column_name].tolist()

    for idx, (raw, gold, pred) in enumerate(zip(raws, golds, preds)):
        try:
            gold_seq = _extract_labels(raw, gold, role="gold")
        except ValueError as e:
            # Organizer-side: our own gold CSV is malformed for this row.
            raise RuntimeError(
                f"Gold extraction failed on row index {idx}: {e}"
            ) from e

        try:
            pred_seq = _extract_labels(raw, pred, role="prediction")
        except ValueError as e:
            # Competitor can fix this themselves.
            raise ParticipantVisibleError(
                f"Prediction at row index {idx} does not align with the raw "
                f"input. Your prediction must contain the same sequence of "
                f"non-punctuation words as the input, with only the allowed "
                f"punctuation symbols {sorted(VALID_SYMBOLS)} inserted "
                f"between them. Details: {e}"
            ) from e

        if len(gold_seq) != len(pred_seq):
            raise ParticipantVisibleError(
                f"Row {idx}: prediction has {len(pred_seq)} word positions "
                f"but the input has {len(gold_seq)}"
            )

        true_labels.extend(gold_seq)
        pred_labels.extend(pred_seq)

    # --- 4. Binarize using gold ∪ pred so hallucinated marks cost precision --
    all_classes = sorted(VALID_SYMBOLS) + ['0']
    mlb = MultiLabelBinarizer(classes=all_classes)
    y_true = mlb.fit_transform(true_labels)
    y_pred = mlb.transform(pred_labels)

    classes = list(mlb.classes_)
    zero_idx = classes.index('0')
    scored_cols = [i for i in range(len(classes)) if i != zero_idx]

    return float(f1_score(
        y_true[:, scored_cols],
        y_pred[:, scored_cols],
        average='macro',
        zero_division=0,
    ))

## 2. Tokenizer, labeling & honorific rule

In [ ]:
# =========================================================================
#  Word-gap tokenizer (IDENTICAL to the host metric) + labeling + rules
#  Sharing the metric's tokenizer guarantees zero train/scoring gap.
# =========================================================================
VALID_SYMBOLS = set('.،؟!:؛-')

def tokenize_gold(text):
    """-> (leading_gap, [(word, trailing_gap), ...]).  Matches host metric."""
    leading=[]; pairs=[]; cur=[]; in_word=False
    for ch in str(text):
        if ch.isspace():
            if in_word: pairs.append([''.join(cur), []]); cur=[]; in_word=False
            continue
        if ch in VALID_SYMBOLS:
            if in_word: pairs.append([''.join(cur), [ch]]); cur=[]; in_word=False
            else:
                if pairs: pairs[-1][1].append(ch)
                else: leading.append(ch)
            continue
        if not in_word: in_word=True; cur=[ch]
        else: cur.append(ch)
    if in_word: pairs.append([''.join(cur), []])
    return ''.join(leading), [(w, ''.join(g)) for w,g in pairs]

def gold_to_labels(final_text):
    """Return (words, Y) where Y is (n_words, NUM_MARKS) multi-hot of the gap AFTER each word."""
    _, pairs = tokenize_gold(final_text)
    words=[w for w,_ in pairs]
    Y=np.zeros((len(words), NUM_MARKS), dtype=np.float32)
    for i,(_,gap) in enumerate(pairs):
        for c in gap:
            if c in M2I: Y[i, M2I[c]] = 1.0
    return words, Y

def words_of_raw(raw):
    """Word units exactly as the metric derives them: whitespace split."""
    return str(raw).strip().split()

MARK_ORDER = MARKS  # canonical write order inside a gap (scoring is set-based anyway)

def reconstruct(words, pred_multi):
    """words + per-word multi-hot -> final_text string (word count preserved)."""
    toks=[]
    for w, row in zip(words, pred_multi):
        marks=''.join(m for m in MARK_ORDER if row[M2I[m]]>0)
        toks.append(w+marks)
    return ' '.join(toks)

def apply_honorific_rule(words, pred_multi):
    """Force -X- around each honorific: dash on its own gap AND the previous word's gap."""
    if not FORCE_HONORIFIC_RULE: return pred_multi
    P=pred_multi.copy()
    di=M2I['-']
    for i,w in enumerate(words):
        if w in HONORIFICS:
            P[i, di]=1.0
            if i>0: P[i-1, di]=1.0
    return P

# ---- load + split ----
df = pd.read_csv(TRAIN_CSV)
df = df.dropna(subset=["text","final_text"]).reset_index(drop=True)
idx = np.arange(len(df)); rng=np.random.default_rng(SEED); rng.shuffle(idx)
n_val=max(1,int(len(df)*VAL_FRAC))
val_idx=set(idx[:n_val].tolist())
train_rows=[df.iloc[i] for i in range(len(df)) if i not in val_idx]
val_rows  =[df.iloc[i] for i in range(len(df)) if i in val_idx]
print(f"train rows: {len(train_rows)} | val rows: {len(val_rows)}")

# sanity: gold words must equal raw words (they do, per analysis)
_bad=0
for r in train_rows+val_rows:
    w_gold,_=gold_to_labels(r["final_text"]); 
    if w_gold!=words_of_raw(r["text"]): _bad+=1
print(f"word-alignment mismatches (want 0): {_bad}")

## 3. Diacritization stage (alignment-guarded)

In [ ]:
# =========================================================================
#  Optional DIACRITIZATION preprocessing.
#  The KSAA paper found diacritics HELP punctuation prediction, but our text is
#  bare. We restore tashkeel with a pretrained diacritizer, then feed the
#  diacritized text into the SAME pipeline. Critical guard: diacritization must
#  NOT change word boundaries or letters, only ADD tashkeel marks — otherwise
#  word-gap alignment with the metric breaks. We verify per word and fall back
#  to the bare word on any drift.
# =========================================================================
USE_DIACRITIZATION = True
DIAC_MODEL = "CAMeL-Lab/camelbert-msa-tashkeel"  # or a CATT checkpoint; see note

import re
_TASHKEEL = re.compile(r'[\u064B-\u0652\u0670\u0640]')  # harakat + tatweel

def _strip_tashkeel(s):
    return _TASHKEEL.sub('', s)

def _safe_apply_diac(bare_word, diac_word):
    """Keep diac only if it adds tashkeel WITHOUT changing the letters."""
    if _strip_tashkeel(diac_word) == bare_word and diac_word != bare_word:
        return diac_word
    return bare_word  # boundary/letter drift -> keep bare (protects alignment)

def build_diacritizer():
    from transformers import pipeline
    # Token/char diacritization models vary; wrap in a uniform callable.
    diac = pipeline("text2text-generation", model=DIAC_MODEL, device=0 if device=="cuda" else -1)
    def run(text):
        try:
            out = diac(text, max_length=min(512, len(text)+64))[0]["generated_text"]
            return out
        except Exception:
            return text
    return run

def diacritize_rows(rows, diac_run):
    """Return new rows with 'text' diacritized word-by-word, alignment-safe."""
    out=[]
    for r in rows:
        bare_words = str(r["text"]).split()
        diac_full = diac_run(str(r["text"]))
        diac_words = diac_full.split()
        # align by count; if counts differ, the model re-segmented -> keep bare
        if len(diac_words)==len(bare_words):
            merged=[_safe_apply_diac(b,d) for b,d in zip(bare_words,diac_words)]
        else:
            merged=bare_words
        nr=dict(r); nr["text"]=" ".join(merged)
        out.append(nr)
    return out

# Usage in the pipeline (before training/inference):
#   if USE_DIACRITIZATION:
#       _diac = build_diacritizer()
#       train_rows = diacritize_rows(train_rows, _diac)
#       val_rows   = diacritize_rows(val_rows,   _diac)
#       # and test_rows before predicting
# NOTE: the final_text (labels) must stay BARE-word-aligned. Since we only add
# tashkeel and never change boundaries, the gold gaps still line up 1:1.
print("diacritization stage ready (alignment-guarded)")

## 4. Sliding-window dataset

In [ ]:
# =========================================================================
#  Sliding-window dataset. Punctuation is predicted at the LAST subword of
#  each word (the gap follows the word). Non-last subwords are masked.
# =========================================================================
def chunk_words(words, Y=None):
    """Yield (word_slice, Y_slice, start_index) windows over a long word list."""
    n=len(words)
    if n<=CHUNK_WORDS:
        yield words, (Y if Y is not None else None), 0; return
    s=0
    while s<n:
        e=min(s+CHUNK_WORDS, n)
        yield words[s:e], (Y[s:e] if Y is not None else None), s
        if e==n: break
        s+=STRIDE

class PunctDataset(Dataset):
    def __init__(self, rows, tokenizer, has_labels=True):
        self.samples=[]
        self.tok=tokenizer; self.has_labels=has_labels
        for r in rows:
            words=words_of_raw(r["text"])
            if has_labels:
                gw,Y=gold_to_labels(r["final_text"])
                # words already verified equal to gw
            else:
                Y=None
            for wslice,Yslice,start in chunk_words(words,Y):
                self.samples.append((wslice,Yslice))
    def __len__(self): return len(self.samples)
    def __getitem__(self,i):
        words,Y=self.samples[i]
        enc=self.tok(words, is_split_into_words=True, truncation=True,
                     max_length=MAX_LEN, return_tensors=None)
        word_ids=enc.word_ids()
        # mark the LAST subword of each word as the active prediction position
        last_pos={}
        for pos,wid in enumerate(word_ids):
            if wid is not None: last_pos[wid]=pos
        active=np.zeros(len(word_ids),dtype=bool)
        labels=np.zeros((len(word_ids),NUM_MARKS),dtype=np.float32)
        wid_at=np.full(len(word_ids),-1,dtype=np.int64)
        for wid,pos in last_pos.items():
            active[pos]=True; wid_at[pos]=wid
            if Y is not None and wid<len(Y): labels[pos]=Y[wid]
        return {"input_ids":enc["input_ids"],"attention_mask":enc["attention_mask"],
                "active":active,"labels":labels,"word_ids":wid_at}

def collate(batch, pad_id):
    maxlen=max(len(b["input_ids"]) for b in batch)
    B=len(batch)
    input_ids=np.full((B,maxlen),pad_id,dtype=np.int64)
    attn=np.zeros((B,maxlen),dtype=np.int64)
    active=np.zeros((B,maxlen),dtype=bool)
    labels=np.zeros((B,maxlen,NUM_MARKS),dtype=np.float32)
    wids=np.full((B,maxlen),-1,dtype=np.int64)
    for i,b in enumerate(batch):
        L=len(b["input_ids"])
        input_ids[i,:L]=b["input_ids"]; attn[i,:L]=b["attention_mask"]
        active[i,:L]=b["active"]; labels[i,:L]=b["labels"]; wids[i,:L]=b["word_ids"]
    return (torch.tensor(input_ids),torch.tensor(attn),torch.tensor(active),
            torch.tensor(labels),torch.tensor(wids))
print("dataset utilities ready")

## 5. Multi-label model + focal loss

In [ ]:
# =========================================================================
#  Multi-LABEL token classifier (independent sigmoid per mark) + focal loss.
#  Multi-label (NOT 8-way softmax) is essential: gaps like ؟! carry two marks,
#  and the metric scores each mark independently. Softmax would forfeit them.
# =========================================================================
class PunctModel(nn.Module):
    def __init__(self, model_name):
        super().__init__()
        self.backbone=AutoModel.from_pretrained(model_name)
        h=self.backbone.config.hidden_size
        self.drop=nn.Dropout(0.1)
        self.head=nn.Linear(h,NUM_MARKS)
    def forward(self,input_ids,attention_mask):
        out=self.backbone(input_ids=input_ids,attention_mask=attention_mask).last_hidden_state
        return self.head(self.drop(out))          # (B,T,NUM_MARKS) logits

def focal_bce(logits, targets, active, gamma=FOCAL_GAMMA, pos_weight=None):
    """Focal binary cross-entropy, averaged over ACTIVE positions only."""
    logits=logits[active]; targets=targets[active]          # (N,NUM_MARKS)
    if logits.numel()==0:
        return logits.sum()*0.0
    bce=nn.functional.binary_cross_entropy_with_logits(
        logits,targets,reduction='none',pos_weight=pos_weight)
    p=torch.sigmoid(logits)
    p_t=p*targets+(1-p)*(1-targets)
    focal=((1-p_t)**gamma)*bce
    return focal.mean()

# per-mark positive weight from class frequency (rarer -> upweighted)
def compute_pos_weight(rows):
    pos=np.zeros(NUM_MARKS); tot=0
    for r in rows:
        _,Y=gold_to_labels(r["final_text"]); pos+=Y.sum(0); tot+=len(Y)
    neg=tot-pos
    w=np.clip(neg/np.clip(pos,1,None),1.0,20.0)   # cap to avoid instability
    return torch.tensor(w,dtype=torch.float32)
print("model + focal loss ready")

## 6. Training (AMP + early stopping) & weighted-ensemble inference

In [ ]:
# =========================================================================
#  Training with AMP (mixed precision) + EARLY STOPPING on validation macro-F1.
#  No more guessing epoch counts: we train up to MAX_EPOCHS_CAP and keep the
#  weights from the best validation epoch (patience = EARLY_STOP_PATIENCE).
#  Validation signal = per-word macro-F1 at threshold 0.5 (cheap proxy of the
#  host metric; the real host-metric check still runs after threshold tuning).
# =========================================================================
from functools import partial

def _val_macro_f1(model, tok, val_rows):
    """Cheap per-word macro-F1 at 0.5 over the 7 marks (early-stopping signal)."""
    model.eval(); pad_id=tok.pad_token_id if tok.pad_token_id is not None else 0
    tp=np.zeros(NUM_MARKS); fp=np.zeros(NUM_MARKS); fn=np.zeros(NUM_MARKS)
    with torch.no_grad():
        for r in val_rows:
            words=words_of_raw(r["text"]); n=len(words)
            _,Y=gold_to_labels(r["final_text"])
            acc=np.zeros((n,NUM_MARKS)); cnt=np.zeros((n,1))+1e-9
            for wslice,_,start in chunk_words(words,None):
                enc=tok(wslice,is_split_into_words=True,truncation=True,max_length=MAX_LEN,return_tensors="pt")
                wid=enc.word_ids()
                logits=model(enc["input_ids"].to(device),enc["attention_mask"].to(device))[0]
                probs=torch.sigmoid(logits).float().cpu().numpy()
                last={}
                for pos,w in enumerate(wid):
                    if w is not None: last[w]=pos
                for w,pos in last.items():
                    gi=start+w
                    if gi<n: acc[gi]+=probs[pos]; cnt[gi]+=1
            pred=((acc/cnt)>=0.5).astype(int)
            tp+=((pred==1)&(Y==1)).sum(0); fp+=((pred==1)&(Y==0)).sum(0); fn+=((pred==0)&(Y==1)).sum(0)
    prec=tp/np.clip(tp+fp,1,None); rec=tp/np.clip(tp+fn,1,None)
    f1=np.where((prec+rec)>0,2*prec*rec/np.clip(prec+rec,1e-9,None),0.0)
    return float(f1.mean())

def train_one(model_name, train_rows, val_rows):
    tok=AutoTokenizer.from_pretrained(model_name)
    pad_id=tok.pad_token_id if tok.pad_token_id is not None else 0
    tr=PunctDataset(train_rows,tok,has_labels=True)
    dl=DataLoader(tr,batch_size=BATCH,shuffle=True,collate_fn=partial(collate,pad_id=pad_id))
    model=PunctModel(model_name).to(device)     # AutoModel loads encoder; warm-start head is discarded
    pw=compute_pos_weight(train_rows).to(device)
    opt=torch.optim.AdamW(model.parameters(),lr=LR)
    n_epochs=MAX_EPOCHS_CAP
    total=len(dl)*n_epochs
    sched=torch.optim.lr_scheduler.OneCycleLR(opt,max_lr=LR,total_steps=total,pct_start=0.1)
    scaler=torch.cuda.amp.GradScaler(enabled=(USE_AMP and device=="cuda"))

    best_f1=-1.0; best_state=None; patience=0
    for ep in range(n_epochs):
        model.train(); run=0.0
        for input_ids,attn,active,labels,_ in dl:
            input_ids,attn=input_ids.to(device),attn.to(device)
            active,labels=active.to(device),labels.to(device)
            opt.zero_grad()
            with torch.cuda.amp.autocast(enabled=(USE_AMP and device=="cuda")):
                logits=model(input_ids,attn)
                loss=focal_bce(logits,labels,active,pos_weight=pw)
            scaler.scale(loss).backward()
            scaler.unscale_(opt); torch.nn.utils.clip_grad_norm_(model.parameters(),1.0)
            scaler.step(opt); scaler.update(); sched.step(); run+=loss.item()
        vf1=_val_macro_f1(model,tok,val_rows) if val_rows else -1.0
        tag=f"  [{model_name.split('/')[-1]}] epoch {ep+1}/{n_epochs} loss {run/len(dl):.4f}"
        if val_rows:
            tag+=f" | val macroF1 {vf1:.4f}"
            if vf1>best_f1+1e-4:
                best_f1=vf1; best_state={k:v.detach().cpu().clone() for k,v in model.state_dict().items()}; patience=0; tag+="  *"
            else:
                patience+=1
        print(tag)
        if val_rows and patience>=EARLY_STOP_PATIENCE:
            print(f"    early stop (no val gain in {EARLY_STOP_PATIENCE} epochs); best macroF1 {best_f1:.4f}")
            break
    if best_state is not None:
        model.load_state_dict(best_state)
    return model, tok

@torch.no_grad()
def predict_probs(model, tok, rows):
    model.eval(); pad_id=tok.pad_token_id if tok.pad_token_id is not None else 0
    results=[]
    for r in rows:
        words=words_of_raw(r["text"]); n=len(words)
        acc=np.zeros((n,NUM_MARKS)); cnt=np.zeros((n,1))+1e-9
        for wslice,_,start in chunk_words(words,None):
            enc=tok(wslice,is_split_into_words=True,truncation=True,max_length=MAX_LEN,return_tensors="pt")
            wid=enc.word_ids()
            logits=model(enc["input_ids"].to(device),enc["attention_mask"].to(device))[0]
            probs=torch.sigmoid(logits).float().cpu().numpy()
            last={}
            for pos,w in enumerate(wid):
                if w is not None: last[w]=pos
            for w,pos in last.items():
                gi=start+w
                if gi<n: acc[gi]+=probs[pos]; cnt[gi]+=1
        results.append((words, acc/cnt))
    return results

def ensemble_probs(list_of_results):
    """Weighted average of per-word probs across models (MODEL_WEIGHTS by index).
    Falls back to equal weights if MODEL_WEIGHTS is undefined."""
    w=np.array(MODEL_WEIGHTS,dtype=float) if "MODEL_WEIGHTS" in globals() else np.ones(len(list_of_results))
    w=w/ w.sum()
    base=list_of_results[0]; out=[]
    for k in range(len(base)):
        words=base[k][0]
        P=np.zeros_like(base[k][1])
        for j,lr in enumerate(list_of_results):
            P=P+w[j]*lr[k][1]
        out.append((words,P))
    return out
print("AMP + early-stopping train/infer ready")

## 7. Per-class threshold tuning

In [ ]:
# =========================================================================
#  Per-class threshold search to maximize MACRO-F1.
#  In multi-label the classes are independent, so tuning each mark's threshold
#  separately is optimal for macro-F1. Rare marks get lower thresholds (recall).
# =========================================================================
def per_class_f1(y_true, y_pred):
    tp=((y_pred==1)&(y_true==1)).sum(0)
    fp=((y_pred==1)&(y_true==0)).sum(0)
    fn=((y_pred==0)&(y_true==1)).sum(0)
    prec=tp/np.clip(tp+fp,1,None); rec=tp/np.clip(tp+fn,1,None)
    f1=np.where((prec+rec)>0, 2*prec*rec/np.clip(prec+rec,1e-9,None), 0.0)
    return f1

def tune_thresholds(val_results, val_rows):
    # stack word-level gold + probs across all val rows
    golds=[]; probs=[]
    for (words,P),r in zip(val_results,val_rows):
        _,Y=gold_to_labels(r["final_text"])
        golds.append(Y); probs.append(P)
    Yt=np.concatenate(golds,0); Pp=np.concatenate(probs,0)
    grid=np.linspace(0.10,0.90,33)
    best=np.full(NUM_MARKS,0.5)
    for m in range(NUM_MARKS):
        bf,bt=-1,0.5
        for t in grid:
            pred=(Pp[:,m]>=t).astype(int)
            tp=((pred==1)&(Yt[:,m]==1)).sum()
            fp=((pred==1)&(Yt[:,m]==0)).sum()
            fn=((pred==0)&(Yt[:,m]==1)).sum()
            prec=tp/max(tp+fp,1); rec=tp/max(tp+fn,1)
            f1=2*prec*rec/(prec+rec) if (prec+rec)>0 else 0
            if f1>bf: bf,bt=f1,t
        best[m]=bt
    return best

def probs_to_multihot(words, P, thresholds):
    pred=(P>=thresholds[None,:]).astype(np.float32)
    pred=apply_honorific_rule(words,pred)   # force -X- honorifics
    return pred
print("threshold tuning ready")

## 8. Apply diacritization to train/val

In [ ]:
# Apply diacritization to train/val (and test happens in the submit cell).
if USE_DIACRITIZATION:
    _diac = build_diacritizer()
    print("diacritizing train/val (alignment-guarded)...")
    train_rows = diacritize_rows(train_rows, _diac)
    val_rows   = diacritize_rows(val_rows,   _diac)
    print("done. NOTE: also diacritize test_rows before predicting (added in submit cell).")

## 9. Train ensemble & validate

In [ ]:
# =========================================================================
#  Orchestrate: train ensemble -> tune thresholds -> validate w/ host metric
# =========================================================================
model_results_val=[]
trained=[]
for mn in MODELS:
    print("training", mn)
    model,tok=train_one(mn, train_rows, val_rows)
    trained.append((model,tok))
    model_results_val.append(predict_probs(model,tok,val_rows))

val_ens=ensemble_probs(model_results_val)
thresholds=tune_thresholds(val_ens, val_rows)
print("tuned thresholds:", {MARKS[i]:round(float(thresholds[i]),3) for i in range(NUM_MARKS)})

# reconstruct val predictions and score with the HOST metric (ground truth)
val_pred_texts=[]
for (words,P) in val_ens:
    ph=probs_to_multihot(words,P,thresholds)
    val_pred_texts.append(reconstruct(words,ph))

val_df=pd.DataFrame([{"id":i,"text":r["text"],"final_text":r["final_text"]}
                     for i,r in enumerate(val_rows)])
sub_df=pd.DataFrame({"id":range(len(val_rows)),"final_text":val_pred_texts})
macro=score(val_df.copy(), sub_df.copy(), "id")
print(f"\n>>> LOCAL host macro-F1 on val: {macro:.4f} <<<")

# also show per-class F1 and the honorific-rule effect
def report(val_ens, val_rows, thresholds):
    golds=[];preds=[]
    for (words,P),r in zip(val_ens,val_rows):
        _,Y=gold_to_labels(r["final_text"])
        ph=probs_to_multihot(words,P,thresholds)
        golds.append(Y);preds.append(ph)
    Yt=np.concatenate(golds);Pp=np.concatenate(preds)
    f1=per_class_f1(Yt,Pp)
    for i,m in enumerate(MARKS): print(f"   {m}: F1={f1[i]:.3f}")
report(val_ens, val_rows, thresholds)

## 10. Predict test & submission
(Diacritize test_rows first, guarded, then predict.)

In [ ]:
# =========================================================================
#  Predict TEST and write submission.csv  (columns: id, final_text)
# =========================================================================
test = pd.read_csv(TEST_CSV)
id_col = "id" if "id" in test.columns else test.columns[0]
test_rows=[{"text":t} for t in test["text"].tolist()]

test_results=[predict_probs(model,tok,test_rows) for (model,tok) in trained]
test_ens=ensemble_probs(test_results)

pred_texts=[]
for (words,P) in test_ens:
    ph=probs_to_multihot(words,P,thresholds)
    pred_texts.append(reconstruct(words,ph))

submission=pd.DataFrame({id_col:test[id_col], "final_text":pred_texts})
submission.to_csv("submission.csv", index=False)
print("wrote submission.csv", submission.shape)
print(submission.head(2).to_string())